In [1]:
# Core imports for the whole lab
import numpy as np
import sympy as sp

x, y = sp.symbols('x y')      # symbolic variables we'll reuse
sp.init_printing()            # pretty-print symbolic math
np.random.seed(42)
print('Setup complete. SymPy', sp.__version__, '| NumPy', np.__version__)


Setup complete. SymPy 1.14.0 | NumPy 2.0.2


In [2]:
# -----------------------------------------------------------
# 🔹 1A. NUMERICAL DERIVATIVE (finite difference)
# -----------------------------------------------------------

# The derivative is the slope: how much f changes for a tiny step h
def f(x):
    return x**2

h = 1e-6
slope_at_3 = (f(3 + h) - f(3)) / h
print('Numerical f\'(3):', round(slope_at_3, 4), ' (exact = 6)')


Numerical f'(3): 6.0  (exact = 6)


In [3]:
# -----------------------------------------------------------
# 🔹 1B. SYMBOLIC DERIVATIVE (SymPy)
# -----------------------------------------------------------

expr = x**2
deriv = sp.diff(expr, x)          # differentiate w.r.t. x
print('d/dx (x**2) =', deriv)     # 2*x

# Evaluate the symbolic derivative at x = 3
print('Symbolic f\'(3) =', deriv.subs(x, 3))

d/dx (x**2) = 2*x
Symbolic f'(3) = 6


In [4]:
import sympy as sp

def g(x):
    return x**3 + 2*x

# 1. Numerical derivative at x = 2 (use h = 1e-6)
h = 1e-6
x0 = 2
numerical_derivative = (g(x0 + h) - g(x0)) / h
print("Numerical derivative at x = 2:", numerical_derivative)

# 2. Symbolic derivative of x**3 + 2*x
x = sp.symbols('x')
g_sym = x**3 + 2*x
symbolic_derivative = sp.diff(g_sym, x)
print("Symbolic derivative:", symbolic_derivative)

# 3. Evaluate the symbolic derivative at x = 2 and compare
symbolic_value = symbolic_derivative.subs(x, 2)
print("Symbolic derivative at x = 2:", symbolic_value)

print("Difference:", abs(numerical_derivative - float(symbolic_value)))

Numerical derivative at x = 2: 14.000006002490295
Symbolic derivative: 3*x**2 + 2
Symbolic derivative at x = 2: 14
Difference: 6.002490295031748e-06


In [5]:
# -----------------------------------------------------------
# 🔹 2A. PARTIAL DERIVATIVES
# -----------------------------------------------------------

f2 = x**2 + 3*x*y + y**2

# A partial derivative differentiates ONE variable, holding others fixed
print('df/dx =', sp.diff(f2, x))     # 2*x + 3*y
print('df/dy =', sp.diff(f2, y))     # 3*x + 2*y


df/dx = 2*x + 3*y
df/dy = 3*x + 2*y


In [6]:

# -----------------------------------------------------------
# 🔹 2B. THE GRADIENT (vector of partials)
# -----------------------------------------------------------

# The gradient stacks every partial derivative into one vector
grad = [sp.diff(f2, v) for v in (x, y)]
print('grad f =', grad)

# Evaluate the gradient at the point (x=1, y=2)
grad_at = [g.subs({x: 1, y: 2}) for g in grad]
print('grad f at (1, 2) =', grad_at)   # points in the steepest-ascent direction



grad f = [2*x + 3*y, 3*x + 2*y]
grad f at (1, 2) = [8, 7]


In [7]:
import sympy as sp

# Define symbols
x, y = sp.symbols('x y')

h2 = x**2 * y + sp.sin(y)

# 1. dh/dx and dh/dy
dh_dx = sp.diff(h2, x)
dh_dy = sp.diff(h2, y)

print("dh/dx =", dh_dx)
print("dh/dy =", dh_dy)

# 2. Assemble the gradient list
gradient = [dh_dx, dh_dy]
print("Gradient =", gradient)

# 3. Evaluate at (x=2, y=0)
gradient_at_point = [expr.subs({x: 2, y: 0}) for expr in gradient]
print("Gradient at (2,0) =", gradient_at_point)

dh/dx = 2*x*y
dh/dy = x**2 + cos(y)
Gradient = [2*x*y, x**2 + cos(y)]
Gradient at (2,0) = [0, 5]


In [8]:
# -----------------------------------------------------------
# 🔹 3A. CHAIN RULE BY HAND vs SymPy
# -----------------------------------------------------------

# y = sin(x**2) is a composition: outer = sin(u), inner = u = x**2
# Chain rule:  dy/dx = cos(u) * du/dx = cos(x**2) * 2x
by_hand = sp.cos(x**2) * 2*x
by_sympy = sp.diff(sp.sin(x**2), x)

print('By hand :', by_hand)
print('By SymPy:', by_sympy)
print('Match?  ', sp.simplify(by_hand - by_sympy) == 0)

By hand : 2*x*cos(x**2)
By SymPy: 2*x*cos(x**2)
Match?   True


In [9]:
# -----------------------------------------------------------
# 🔹 3B. CHAINING THREE FUNCTIONS
# -----------------------------------------------------------

# y = (3x + 1)**4  -> outer^4, inner (3x+1)
expr3 = (3*x + 1)**4
print('d/dx (3x+1)^4 =', sp.diff(expr3, x))   # 12*(3x+1)^3

d/dx (3x+1)^4 = 12*(3*x + 1)**3


In [10]:
import sympy as sp

# Define symbol
x = sp.symbols('x')

# y = exp(x**2 + 1)
y = sp.exp(x**2 + 1)

# 1. By hand (using the chain rule)
inner = x**2 + 1
inner_prime = 2 * x
dy_dx_hand = sp.exp(inner) * inner_prime
print("By hand:", dy_dx_hand)

# 2. With sp.diff
dy_dx_diff = sp.diff(y, x)
print("With sp.diff:", dy_dx_diff)

# 3. Confirm they match
print("Do they match?", sp.simplify(dy_dx_hand - dy_dx_diff) == 0)

By hand: 2*x*exp(x**2 + 1)
With sp.diff: 2*x*exp(x**2 + 1)
Do they match? True


In [11]:
# -----------------------------------------------------------
# 🔹 4A. FORWARD PASS
# -----------------------------------------------------------

# Tiny toy problem: 4 samples, 3 input features, 5 hidden units, 1 output
X = np.random.randn(4, 3)
Y = np.random.randn(4, 1)
W1 = np.random.randn(3, 5) * 0.1
W2 = np.random.randn(5, 1) * 0.1

z1 = X @ W1                 # linear layer 1
h  = np.maximum(0, z1)      # ReLU activation
y_hat = h @ W2              # linear layer 2 (prediction)
loss = ((y_hat - Y) ** 2).mean()
print('Initial loss:', round(loss, 4))


Initial loss: 1.6936


In [12]:
# -----------------------------------------------------------
# 🔹 4B. BACKWARD PASS (gradients via the chain rule)
# -----------------------------------------------------------

# Work backwards from the loss, one link at a time
dy   = 2 * (y_hat - Y) / Y.size      # d loss / d y_hat
dW2  = h.T @ dy                      # d loss / d W2
dh   = dy @ W2.T                     # d loss / d h
dz1  = dh * (z1 > 0)                 # ReLU gradient (1 where z1>0 else 0)
dW1  = X.T @ dz1                     # d loss / d W1

print('dW1 shape:', dW1.shape, '(matches W1)')
print('dW2 shape:', dW2.shape, '(matches W2)')


dW1 shape: (3, 5) (matches W1)
dW2 shape: (5, 1) (matches W2)


In [13]:
# -----------------------------------------------------------
# 🔹 4C. ONE GRADIENT-DESCENT STEP SHOULD LOWER THE LOSS
# -----------------------------------------------------------

lr = 0.1
W1 -= lr * dW1               # step downhill
W2 -= lr * dW2

# Recompute the loss after the update
h_new = np.maximum(0, X @ W1)
loss_new = ((h_new @ W2 - Y) ** 2).mean()
print('Loss before:', round(loss, 4))
print('Loss after :', round(loss_new, 4), '-> should be lower')


Loss before: 1.6936
Loss after : 1.6524 -> should be lower


In [14]:
import numpy as np

Xb = np.random.randn(6, 4)
Yb = np.random.randn(6, 1)
Wa = np.random.randn(4, 8) * 0.1     # input -> hidden
Wb = np.random.randn(8, 1) * 0.1     # hidden -> output

# ---------------------------------------------------
# 1. Forward pass: z1, h = ReLU(z1), y_hat, loss
# ---------------------------------------------------
z1 = Xb @ Wa
h = np.maximum(0, z1)          # ReLU activation
y_hat = h @ Wb
loss = np.mean((y_hat - Yb) ** 2)

print("Loss before update:", loss)

# ---------------------------------------------------
# 2. Backward pass: dy, dWb, dh, dz1, dWa
# ---------------------------------------------------
m = Xb.shape[0]

dy = (2 / m) * (y_hat - Yb)    # dLoss/dy_hat

dWb = h.T @ dy                 # Gradient of Wb
dh = dy @ Wb.T                 # Backprop to hidden layer
dz1 = dh * (z1 > 0)            # ReLU derivative
dWa = Xb.T @ dz1               # Gradient of Wa

# ---------------------------------------------------
# 3. One step with lr = 0.05; print loss before and after
# ---------------------------------------------------
lr = 0.05

Wa -= lr * dWa
Wb -= lr * dWb

# Forward pass after update
z1 = Xb @ Wa
h = np.maximum(0, z1)
y_hat = h @ Wb
loss_after = np.mean((y_hat - Yb) ** 2)

print("Loss after update:", loss_after)

Loss before update: 0.8782154774250645
Loss after update: 0.871893730342554


In [18]:
import sympy as sp

# Define symbolic variables
x, y = sp.symbols('x y')

# Function
f5 = x**2 + 3*x*y + y**2

# Hessian matrix (matrix of second derivatives)
H = sp.hessian(f5, (x, y))

print("Hessian of f:")
sp.pprint(H)

Hessian of f:
⎡2  3⎤
⎢    ⎥
⎣3  2⎦


In [19]:
# -----------------------------------------------------------
# 🔹 5B. GRADIENT DESCENT ON A SIMPLE FUNCTION
# -----------------------------------------------------------

# Minimise f(x) = (x - 4)**2, whose minimum is at x = 4.
# Update rule:  x <- x - lr * f'(x),  with f'(x) = 2*(x - 4)
xv = 0.0          # starting point
lr = 0.2
for step in range(15):
    grad = 2 * (xv - 4)        # the derivative
    xv = xv - lr * grad        # step against the gradient
print('Converged x:', round(xv, 3), ' (true minimum = 4)')

Converged x: 3.998  (true minimum = 4)


In [20]:
import sympy as sp

# Define symbolic variables
x, y = sp.symbols('x y')

# -----------------------------------------------------------
# 1. Hessian of x**4 + y**2
# -----------------------------------------------------------
f = x**4 + y**2

H = sp.hessian(f, (x, y))

print("Hessian:")
sp.pprint(H)

# -----------------------------------------------------------
# 2. Gradient descent to minimise (x - 7)**2
# -----------------------------------------------------------
xv = 0.0
lr = 0.1

for step in range(20):
    grad = 2 * (xv - 7)      # Derivative of (x - 7)^2
    xv = xv - lr * grad

print("Final x:", xv)

Hessian:
⎡    2   ⎤
⎢12⋅x   0⎥
⎢        ⎥
⎣  0    2⎦
Final x: 6.91929549467752
